# 13 Full Structured Sequence Baseline

Consumes `clips/{FULL_RUN_ID}/clips_v1.parquet` and `manifests/bbe_events_v1.parquet`, then builds structured sequence artifacts, player-season prior rows, predictions, and metrics.

Required inputs:

- `manifests/bbe_events_v1.parquet`
- `clips/{FULL_RUN_ID}/clips_v1.parquet`
- `configs/targets/target_registry_v1.yaml`

Created outputs:

- `features/structured_sequence_v1/manifest.parquet`
- `features/structured_sequence_v1/frames.parquet`
- `features/clip_embedding_v1/manifest.parquet`
- `features/player_season_embedding_v1/manifest.parquet`
- `datasets/sequence_dataset_v1/manifest.parquet`
- `datasets/event_with_player_prior_v1/manifest.parquet`
- `predictions/sequence_structured_mlb_2024_2026_v1/predictions_v1.parquet`
- `predictions/sequence_structured_mlb_2024_2026_v1/metrics_v1.json`

Next: `notebooks/14_full_video_baseline.ipynb`.

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
SEQUENCE_RUN_ID = run_id(RUN_PROFILE, 'sequence_run_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
CLIP_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'clip_embedding_feature_id')
PLAYER_SEASON_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'player_season_embedding_feature_id')
SEQUENCE_DATASET_ID = artifact_id(RUN_PROFILE, 'sequence_dataset_id')
EVENT_WITH_PRIOR_DATASET_ID = artifact_id(RUN_PROFILE, 'event_with_prior_dataset_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('SEQUENCE_RUN_ID =', SEQUENCE_RUN_ID)
print('STRUCTURED_SEQUENCE_FEATURE_ID =', STRUCTURED_SEQUENCE_FEATURE_ID)


## Input Check

If `clips_v1.parquet` is missing, run `12_full_cv_preprocess.ipynb` after real video sources/downloads are available.

In [ ]:
from pathlib import Path
import sys
import importlib.util

DEFAULT_REPO_DIR = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
REPO_DIR = globals().get('REPO_DIR', DEFAULT_REPO_DIR)
BASE_DIR = globals().get('BASE_DIR', Path('/content/drive/MyDrive/baseball_vision'))
FULL_RUN_ID = globals().get('FULL_RUN_ID', 'mlb_2024_2026_full_v2')
src_dir = globals().get('src_dir', REPO_DIR / 'src')

if importlib.util.find_spec('sport_pipeline') is None:
    init_file = src_dir / 'sport_pipeline' / '__init__.py'
    if not init_file.exists():
        raise FileNotFoundError(
            'sport_pipeline package が見つかりません。13 の最初の setup cell を実行するか、Drive の repo 配置を確認してください。\n'
            f'checked init_file={init_file}\n'
            f'src_dir_exists={src_dir.exists()}\n'
            f'src_dir_children={[p.name for p in list(src_dir.iterdir())[:20]] if src_dir.exists() else []}'
        )
    sys.path.insert(0, str(src_dir))

from sport_pipeline.artifact_check import check_artifacts
from sport_pipeline.cv.diagnostics import clip_quality_diagnostics, format_clip_quality_diagnostics, is_clean_trainable_clip
from sport_pipeline.io import read_table

required = [
    'manifests/bbe_events_v1.parquet',
    f'clips/{FULL_RUN_ID}/clips_v1.parquet',
]
print(check_artifacts(BASE_DIR, required))

clips_path = BASE_DIR / 'clips' / FULL_RUN_ID / 'clips_v1.parquet'
clip_rows = read_table(clips_path) if clips_path.exists() else []
clean_rows = [row for row in clip_rows if is_clean_trainable_clip(row)]
diagnostics = clip_quality_diagnostics(clip_rows)
print('clips_v1 rows =', len(clip_rows))
print('clean trainable clip rows =', len(clean_rows))
print(format_clip_quality_diagnostics(diagnostics))
if not clean_rows:
    raise RuntimeError(
        '13 cannot run because clips_v1 has 0 clean trainable rows. '
        'Do not continue to 17/18. Rerun 12 after checking the diagnostics above. '
        'If review_reason is mostly unknown/low view or visibility, 12 did not classify the downloaded videos as batting-form clips. '
        'If this is from an old 12 run, set full_cv.resume=False once or remove the old clips/{FULL_RUN_ID} outputs so 12 recomputes candidates.'
    )


## Build Sequence Artifacts

`PRIOR_MODE='past_only'` is the leakage-safe default. `oracle_full_season` is analysis-only because it can use future clips.

In [ ]:
from sport_pipeline.models.sequence.full_baseline import run_full_sequence_baseline

SEQUENCE_SETTINGS = stage_settings(RUN_PROFILE, 'sequence_baseline')
PRIOR_MODE = str(SEQUENCE_SETTINGS.get('prior_mode', 'past_only'))
AGGREGATION_METHOD = str(SEQUENCE_SETTINGS.get('aggregation_method', 'quality_weighted_pooling'))
FRAME_COUNT = int(SEQUENCE_SETTINGS.get('frame_count', 32))
REQUIRE_NON_EMPTY_SEQUENCE_BASELINE = bool(SEQUENCE_SETTINGS.get('require_non_empty', True))

outputs = run_full_sequence_baseline(
    BASE_DIR,
    clip_run_id=FULL_RUN_ID,
    prediction_run_id=SEQUENCE_RUN_ID,
    prior_mode=PRIOR_MODE,
    aggregation_method=AGGREGATION_METHOD,
    frame_count=FRAME_COUNT,
    require_non_empty=REQUIRE_NON_EMPTY_SEQUENCE_BASELINE,
    structured_sequence_feature_id=STRUCTURED_SEQUENCE_FEATURE_ID,
    clip_embedding_feature_id=CLIP_EMBEDDING_FEATURE_ID,
    player_season_embedding_feature_id=PLAYER_SEASON_EMBEDDING_FEATURE_ID,
    sequence_dataset_id=SEQUENCE_DATASET_ID,
    event_with_prior_dataset_id=EVENT_WITH_PRIOR_DATASET_ID,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


## Output Check

In [ ]:
from sport_pipeline.io import read_table

expected = [
    f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/manifest.parquet',
    f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/frames.parquet',
    f'features/{CLIP_EMBEDDING_FEATURE_ID}/manifest.parquet',
    f'features/{PLAYER_SEASON_EMBEDDING_FEATURE_ID}/manifest.parquet',
    f'datasets/{SEQUENCE_DATASET_ID}/manifest.parquet',
    f'datasets/{EVENT_WITH_PRIOR_DATASET_ID}/manifest.parquet',
    f'predictions/{SEQUENCE_RUN_ID}/predictions_v1.parquet',
    f'predictions/{SEQUENCE_RUN_ID}/metrics_v1.json',
    f'reports/preflight/full_sequence_baseline_{SEQUENCE_RUN_ID}.json',
]
result = check_artifacts(BASE_DIR, expected)
print(result)

sequence_rows = read_table(BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'manifest.parquet')
prior_rows = read_table(BASE_DIR / 'datasets' / EVENT_WITH_PRIOR_DATASET_ID / 'manifest.parquet')
prediction_rows = read_table(BASE_DIR / 'predictions' / SEQUENCE_RUN_ID / 'predictions_v1.parquet')
print('structured_sequence rows =', len(sequence_rows))
print('event_with_player_prior rows =', len(prior_rows))
print('prediction rows =', len(prediction_rows))
if not sequence_rows or not prior_rows or not prediction_rows:
    raise RuntimeError('13 produced empty sequence/prior/prediction artifacts. Stop and fix 12 clips_v1 before continuing.')
print('NEXT: notebooks/14_full_video_baseline.ipynb')
